# **Install required libraries**

In [1]:
!pip install -qU langchain langchain-core langchain-community langchain-groq langchain-google-genai langchain-huggingface langchain-text-splitters langchain-tavily langgraph langgraph-supervisor langgraph-checkpoint-sqlite langsmith chromadb sentence-transformers pypdf pydantic yfinance

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "financial-advisor-capstone"

os.environ["CHROMA_SERVER_NO_TELEMETRY"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# **Agent Fundamentals & LangSmith Observability**

**Define real tools**

In [3]:
from langchain_core.tools import tool
import yfinance as yf

@tool
def get_stock_price(ticker: str) -> dict:
    """Get the current price and recent trend for a given stock ticker."""
    data = yf.Ticker(ticker).history(period="5d")
    return {
        "ticker": ticker,
        "last_close": round(data["Close"].iloc[-1], 2),
        "5day_change_pct": round((data["Close"].iloc[-1] / data["Close"].iloc[0] - 1) * 100, 2),
    }

@tool
def get_company_fundamentals(ticker: str) -> dict:
    """Get key fundamental metrics for a company: P/E ratio, market cap, beta."""
    info = yf.Ticker(ticker).info
    return {
        "pe_ratio": info.get("trailingPE"),
        "market_cap": info.get("marketCap"),
        "beta": info.get("beta"),
    }

**Add a real web search tool**

In [4]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=3)

**Initialize the LLM and bind all tools to it**

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
llm_with_tools = llm.bind_tools([get_stock_price, get_company_fundamentals, search_tool])

**Let the LLM decide which tool to call**

In [6]:
response = llm_with_tools.invoke(
    "What's the current price and P/E ratio of Apple stock, and any recent news about it?"
)
print(response.tool_calls)

[{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'call_324544', 'type': 'tool_call'}, {'name': 'get_company_fundamentals', 'args': {'ticker': 'AAPL'}, 'id': 'call_324545', 'type': 'tool_call'}, {'name': 'tavily_search', 'args': {'query': 'Apple AAPL stock recent news', 'topic': 'finance'}, 'id': 'call_324546', 'type': 'tool_call'}]


**Actually execute the tool calls the model chose**

In [7]:
tool_map = {
    "get_stock_price": get_stock_price,
    "get_company_fundamentals": get_company_fundamentals,
    "tavily_search": search_tool,
}

for call in response.tool_calls:
    result = tool_map[call["name"]].invoke(call["args"])
    print(call["name"], "->", result)

get_stock_price -> {'ticker': 'AAPL', 'last_close': np.float64(304.91), '5day_change_pct': np.float64(-1.87)}
get_company_fundamentals -> {'pe_ratio': 34.966743, 'market_cap': 4449911701504, 'beta': 1.086}
tavily_search -> {'query': 'Apple AAPL stock recent news', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Financial Analysis for AAPL', 'url': 'https://finance.yahoo.com/quote/AAPL/', 'content': 'Stock: AAPL\nFinancial Analysis:\nLatest Open Price: 307.75 Latest Close Price: 304.91 Highest Close Price: 339.79 Lowest Close Price: 171.37 Average Close Price (2 years): 246.20 Standard Deviation of Close Price: 33.40 Volume Traded (2 years): 25961905200 Total Return (2 years): 41.34% Annualized Return (2 years): 18.89% CAGR (2 years): 18.89% Sharpe Ratio (2 years): 0.57 Max Drawdown (2 years): 0.06% ', 'score': 0.98042, 'raw_content': None, 'id': '123e2d-00'}, {'url': 'https://finance.yahoo.com/quote/AAPL/news', 'title': 'Apple Inc. (AAPL) Latest Stock 

**Define a Pydantic schema for structured output**

In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class InvestmentAnalysis(BaseModel):
    ticker: str
    risk_level: Literal["low", "medium", "high"]
    recommendation: Literal["buy", "hold", "sell"]
    reasoning: str = Field(description="Short justification, 1-2 sentences")

**Get structured output from the LLM**

In [9]:
structured_llm = llm.with_structured_output(InvestmentAnalysis)

result = structured_llm.invoke(
    "AAPL has a P/E of 28, beta of 1.2, and rose 3% this week. Give a short investment analysis."
)
print(result)

ticker='AAPL' risk_level='medium' recommendation='hold' reasoning='With a P/E of 28 and a beta of 1.2, Apple shows solid market performance and moderate volatility. The recent 3% weekly gain suggests positive momentum, making it a strong hold for long-term investors.'


# **RAG Pipline**

**Workflow Pattern: Evaluator-Optimizer**

We implement the Evaluator-Optimizer pattern: `evaluate_compliance` acts as the
evaluator, critiquing the proposed allocation against Saudi CMA regulations.
`optimize_strategy` acts as the optimizer, revising the allocation based on
specific violations. This loop iterates up to `MAX_OPTIMIZATION_ROUNDS` times,
converging on a compliant strategy before human approval.

**Upload the regulatory PDFs**



In [10]:
import pathlib
import urllib.request

DATA_DIR = pathlib.Path("data/regulations")
DATA_DIR.mkdir(parents=True, exist_ok=True)

FILES_TO_DOWNLOAD = {
    "GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf": "https://raw.githubusercontent.com/Sarah2433/Tharwa-AI-Powered-Financial-Advisory-Agent/main/data/regulations/GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf",
    "Investment_Funds_Regulations_2025_AR.pdf": "https://raw.githubusercontent.com/Sarah2433/Tharwa-AI-Powered-Financial-Advisory-Agent/main/data/regulations/Investment_Funds_Regulations_2025_AR.pdf",
}

for fname, url in FILES_TO_DOWNLOAD.items():
    dest = DATA_DIR / fname
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
        print(f"Downloaded {fname}")
    else:
        print(f"{fname} already exists, skipping")

print("Files in data/regulations/:")
for f in DATA_DIR.iterdir():
    print(" -", f.name)

GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf already exists, skipping
Investment_Funds_Regulations_2025_AR.pdf already exists, skipping
Files in data/regulations/:
 - Investment_Funds_Regulations_2025_AR.pdf
 - GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf


In [11]:
import os

expected = {
    "GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf": DATA_DIR / "GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf",
    "Investment_Funds_Regulations_2025_AR.pdf": DATA_DIR / "Investment_Funds_Regulations_2025_AR.pdf",
}
for name, path in expected.items():
    if not path.exists():
        print(f"MISSING: {name} -- rename your uploaded file to match, or edit SOURCE_DOCUMENTS below.")

**RAG config**



In [12]:
from pathlib import Path

PERSIST_DIR = Path("chroma_store")
COLLECTION_NAME = "sa_capital_market_regulations"

SOURCE_DOCUMENTS = {
    "capital_market_rules": DATA_DIR / "GeneralSaudiCapitalMarketRulesandRegulationsArabicEdition1Aug23.pdf",
    "investment_funds_regulations": DATA_DIR / "Investment_Funds_Regulations_2025_AR.pdf",
}

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
ARABIC_LEGAL_SEPARATORS = [
    "\nالمادة",   # "Article"
    "\nالباب",    # "Chapter"
    "\nالفصل",    # "Part"
    "\n\n", "\n", "، ", " ", "",
]
RETRIEVAL_K = 4


**Ingest**

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_documents() -> list[Document]:
    all_docs: list[Document] = []
    for source_key, path in SOURCE_DOCUMENTS.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing {path} -- upload it in the cell above.")
        loader = PyPDFLoader(str(path))
        pages = loader.load()
        for page in pages:
            page.metadata["source_document"] = source_key
        all_docs.extend(pages)
        print(f"[ingest] loaded {len(pages)} pages from {path.name}")
    if not all_docs:
        raise RuntimeError("No documents loaded -- check SOURCE_DOCUMENTS.")
    return all_docs


def split_documents(documents: list[Document]) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, separators=ARABIC_LEGAL_SEPARATORS,
    )
    chunks = splitter.split_documents(documents)
    print(f"[ingest] split into {len(chunks)} chunks")
    return chunks


def build_vectorstore(force_rebuild: bool = False) -> Chroma:
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    already_built = PERSIST_DIR.exists() and any(PERSIST_DIR.iterdir())
    if already_built and not force_rebuild:
        print(f"[ingest] loading existing store from {PERSIST_DIR}")
        return Chroma(collection_name=COLLECTION_NAME, embedding_function=embeddings,
                       persist_directory=str(PERSIST_DIR))

    documents = load_documents()
    chunks = split_documents(documents)
    PERSIST_DIR.mkdir(parents=True, exist_ok=True)
    store = Chroma.from_documents(documents=chunks, embedding=embeddings,
                                   collection_name=COLLECTION_NAME, persist_directory=str(PERSIST_DIR))
    store.persist()
    print(f"[ingest] embedded and persisted {len(chunks)} chunks to {PERSIST_DIR}")
    return store


vectorstore = build_vectorstore()


/tmp/ipykernel_39846/2709304308.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[ingest] loading existing store from chroma_store


/tmp/ipykernel_39846/2709304308.py:38: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(collection_name=COLLECTION_NAME, embedding_function=embeddings,


**Retriever + Agentic RAG tool**



**RAG Design Choice: Agentic RAG**

We use Agentic RAG rather than a fixed 2-Step chain: the LLM itself decides,
via `search_regulations` as a bound tool, whether and how many times to query
the regulatory corpus, and with which phrasing. This matters because not every
proposed allocation triggers the same regulatory questions — a 100% equities
plan needs a different lookup than a fund-structure question. A 2-Step chain
would force one fixed retrieval per query regardless of what the strategy
actually needs. We avoid a Hybrid approach here since our corpus is small
(2 documents) and doesn't need a routing layer between multiple retrieval
strategies.

In [14]:
from langchain_core.tools import tool


def get_retriever(k: int = RETRIEVAL_K, source_document: str | None = None):
    search_kwargs = {"k": k}
    if source_document:
        search_kwargs["filter"] = {"source_document": source_document}
    return vectorstore.as_retriever(search_kwargs=search_kwargs)


@tool
def search_regulations(query: str, source_document: str | None = None) -> str:
    """Search the Saudi CMA regulatory corpus for text relevant to `query`.

    Use this to check whether a proposed portfolio strategy (asset types,
    concentration limits, investor eligibility, fund structure, etc.) is
    consistent with Saudi capital market regulations before it is approved.
    Call it more than once with different phrasings if the first search
    doesn't surface the specific rule you need.

    Args:
        query: A specific question, e.g. "concentration limits for a
            single issuer in an investment fund".
        source_document: Optional filter -- "capital_market_rules" or
            "investment_funds_regulations". Leave empty to search both.
    """
    retriever = get_retriever(source_document=source_document)
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant regulatory text was found for this query."
    formatted = []
    for i, d in enumerate(docs, start=1):
        src = d.metadata.get("source_document", "unknown")
        page = d.metadata.get("page", "?")
        formatted.append(f"[{i}] (source: {src}, page: {page})\n{d.page_content.strip()}")
    return "\n\n".join(formatted)


**Prove retrieval actually works**



In [15]:
TEST_QUERIES = [
    ("ما هو تعريف صندوق الاستثمار؟", "investment_funds_regulations"),
    ("ما هي أحكام الالتزام باللائحة؟", "investment_funds_regulations"),
    ("متطلبات صندوق الاستثمار العام", None),
]

for query, source in TEST_QUERIES:
    print("=" * 80)
    print(f"QUERY: {query}  (source filter: {source or 'both documents'})")
    print("-" * 80)
    result = search_regulations.invoke({"query": query, "source_document": source})
    print(result[:800])
    print()


QUERY: ما هو تعريف صندوق الاستثمار؟  (source filter: investment_funds_regulations)
--------------------------------------------------------------------------------
[1] (source: investment_funds_regulations, page: 166)
الآتي: 
أ) وصف لنوع (أو أنواع) الأصول والاستثمارات التي سوف يستثمر فيها الصندوق 
(حيثما ينطبق). 
ب) أي سياسة ينتج عنها تركز الاستثمار في أصول من نوع معين أو منطقة جغرافية 
محددة. 
ج) بيانات صك ملكية العقار/ أو العقارات محل المشروع (حيثما ينطبق).

[2] (source: investment_funds_regulations, page: 114)
114 
 
 
Internal - داخل 
3) سياسات الاستثمار وممارساته  
أ) الأهداف الاستثمارية لصندوق الاستثمار. 
ب)  نوع (أنواع) الأوراق المالية التي سوف يستثمر الصندوق فيها بشكل أساسي. 
ج) أي سياسة لتركيز الاستثمار في أوراق مالية معنية، أو في صناعة أو مجموعة من 
القطاعات، أو في بلد معين أو منطقة جغرافية معينة، على أن تشتمل على الحد الأدنى 
والأقصى لتلك الأوراق المالية. 
د) جدول يوضح نسبة الاستثمار في كل مجال استثماري بحدِّه الأدنى والأعلى.  
ه)     بيان

QUERY: ما هي أحكام الالتزام باللائ

**Structured output contracts**

In [16]:
from typing import Literal
from pydantic import BaseModel, Field


class PortfolioStrategy(BaseModel):
    """A proposed personalized investment strategy."""
    asset_allocation: dict[str, float] = Field(
        description="Asset class -> fraction of portfolio, e.g. "
        "{'equities': 0.5, 'sukuk': 0.3, 'cash': 0.2}. Should sum to ~1.0.")
    risk_level: Literal["conservative", "moderate", "aggressive"]
    rationale: str
    iteration: int = 0


class ComplianceVerdict(BaseModel):
    """Output of the Regulatory Compliance Agent (the Evaluator)."""
    compliant: bool
    violations: list[str] = Field(default_factory=list,
        description="Specific, actionable violations found (not a vague restatement).")
    cited_articles: list[str] = Field(default_factory=list,
        description="Article/section references pulled from the retrieved text.")
    confidence: Literal["low", "medium", "high"]
    rationale: str


In [17]:
from langgraph.func import entrypoint, task
from langgraph.types import RetryPolicy
from langgraph.checkpoint.memory import InMemorySaver

MAX_OPTIMIZATION_ROUNDS = 3
SINGLE_ASSET_CLASS_CAP = 0.60


@task
def generate_strategy(risk_profile: str, investment_horizon_years: int) -> PortfolioStrategy:
    if risk_profile == "conservative":
        allocation = {"sukuk": 0.55, "equities": 0.25, "cash": 0.20}
    elif risk_profile == "aggressive":
        allocation = {"equities": 0.75, "sukuk": 0.15, "cash": 0.10}
    else:
        allocation = {"equities": 0.50, "sukuk": 0.35, "cash": 0.15}

    if investment_horizon_years >= 10:
        shift = min(0.10, allocation["cash"])
        allocation["cash"] -= shift
        allocation["equities"] += shift

    return PortfolioStrategy(
        asset_allocation=allocation, risk_level=risk_profile,
        rationale=f"Initial allocation for a {risk_profile} investor with a {investment_horizon_years}-year horizon.",
        iteration=0,
    )


@task(retry=RetryPolicy(max_attempts=3, initial_interval=1.0, backoff_factor=2.0))
def evaluate_compliance(strategy: PortfolioStrategy, llm) -> ComplianceVerdict:
    tool_llm = llm.bind_tools([search_regulations])
    structured_llm = llm.with_structured_output(ComplianceVerdict)

    scratchpad = (
        "You are the Regulatory Compliance Agent for a Saudi investment advisory system. "
        "Use the search_regulations tool to check this proposed allocation against Saudi CMA rules.\n\n"
        f"Proposed allocation: {strategy.asset_allocation}\n"
        f"Risk level: {strategy.risk_level}\n"
    )

    tool_call_response = tool_llm.invoke(scratchpad)
    retrieved_context = []
    for call in getattr(tool_call_response, "tool_calls", []) or []:
        if call["name"] == "search_regulations":
            result = search_regulations.invoke(call["args"])
            retrieved_context.append(result)
            print("handoff -> search_regulations tool call:", call["args"])

    grounded_prompt = (
        scratchpad + "\n\nRetrieved regulatory text:\n"
        + ("\n\n".join(retrieved_context) if retrieved_context else "(no tool call made)")
        + "\n\nRule on compliance now, citing specific articles from the retrieved text. "
        "If nothing relevant was retrieved, say so in the rationale and set confidence to 'low'."
    )
    return structured_llm.invoke(grounded_prompt)


@task
def optimize_strategy(strategy: PortfolioStrategy, verdict: ComplianceVerdict) -> PortfolioStrategy:
    new_allocation = dict(strategy.asset_allocation)
    for violation in verdict.violations:
        text = violation.lower()
        for asset, pct in list(new_allocation.items()):
            if asset in text and pct > SINGLE_ASSET_CLASS_CAP:
                excess = pct - SINGLE_ASSET_CLASS_CAP
                new_allocation[asset] = SINGLE_ASSET_CLASS_CAP
                new_allocation["cash"] = new_allocation.get("cash", 0.0) + excess
    return PortfolioStrategy(
        asset_allocation=new_allocation, risk_level=strategy.risk_level,
        rationale=strategy.rationale + f" | Revised after round {strategy.iteration + 1}: " + "; ".join(verdict.violations),
        iteration=strategy.iteration + 1,
    )

/tmp/ipykernel_39846/3257731351.py:30: LangGraphDeprecatedSinceV05: `retry` is deprecated and will be removed. Please use `retry_policy` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  @task(retry=RetryPolicy(max_attempts=3, initial_interval=1.0, backoff_factor=2.0))


# **Context & state management**

**Custom State**

In [18]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal, Optional

class InvestorState(AgentState):
    risk_level: NotRequired[Literal["low", "medium", "high"]]
    investment_goal: NotRequired[str]
    amount: NotRequired[float]
    market_analysis: NotRequired[str]
    financial_analysis: NotRequired[str]
    risk_analysis: NotRequired[str]
    portfolio_allocation: NotRequired[str]
    compliance_check: NotRequired[str]
    current_step: NotRequired[str]

(short-term) — **checkpointer**

In [19]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

(long-term) — **Store**

In [20]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

def remember(user_id: str, key: str, value):
    store.put(("users", user_id), key, {"value": value})

def recall(user_id: str, key: str):
    item = store.get(("users", user_id), key)
    return item.value["value"] if item else None

**record risk profile**

In [21]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def record_risk_profile(
    risk_level: Literal["low", "medium", "high"],
    goal: str,
    runtime: ToolRuntime[None, InvestorState],
) -> Command:
    """Record the user's risk level and investment goal."""
    user_id = runtime.config["configurable"]["thread_id"]

    remember(user_id, "risk_level", risk_level)
    remember(user_id, "investment_goal", goal)

    return Command(
        update={
            "risk_level": risk_level,
            "investment_goal": goal,
            "messages": [
                ToolMessage(
                    content=f"Risk level recorded: {risk_level}, goal: {goal}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

**cross-thread**

In [22]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=[record_risk_profile],
    state_schema=InvestorState,
    checkpointer=checkpointer,
)

config_a = {"configurable": {"thread_id": "user-sara-thread-1"}}
result1 = agent.invoke(
    {"messages": [HumanMessage("I prefer medium risk and my goal is retirement, record it")]},
    config_a
)
for m in result1["messages"]:
    m.pretty_print()

config_b = {"configurable": {"thread_id": "user-sara-thread-2-DIFFERENT"}}
saved_risk = recall("user-sara-thread-1", "risk_level")
print(saved_risk)

================================ Human Message =================================

I prefer medium risk and my goal is retirement, record it
================================== Ai Message ==================================

[]
Tool Calls:
  record_risk_profile (call_86301)
 Call ID: call_86301
  Args:
    risk_level: medium
    goal: retirement
================================= Tool Message =================================
Name: record_risk_profile

Risk level recorded: medium, goal: retirement
================================== Ai Message ==================================

[{'type': 'text', 'text': 'I have successfully recorded your risk profile with a medium risk level and retirement as your investment goal.', 'extras': {'signature': 'EmgKZgERTTIPXHSbssqy2kEn/QYsBK/9Ayllhk0gtQhdfb/E8i3M82Jd5cIWjkdcm4CBKeGX/ZeFE5OrncvNIfijrpvYy4N0wwUsGiqpCMmyb29x2lHt9aUn4agDawq0fZf6Wdp/Sob+BQ=='}}]
medium


# **Multi-agent / Routing**

**Multi-agent / Routing Architecture**

In [23]:
from langchain_core.tools import tool
from typing import Literal

@tool
def propose_allocation(risk_profile: Literal["low", "medium", "high"], investment_horizon_years: int) -> dict:
    """Propose an asset allocation (%) based on risk profile and investment horizon."""
    if risk_profile == "low":
        allocation = {"sukuk": 0.55, "equities": 0.25, "cash": 0.20}
    elif risk_profile == "high":
        allocation = {"equities": 0.75, "sukuk": 0.15, "cash": 0.10}
    else:
        allocation = {"equities": 0.50, "sukuk": 0.35, "cash": 0.15}
    if investment_horizon_years >= 10:
        shift = min(0.10, allocation["cash"])
        allocation["cash"] -= shift
        allocation["equities"] += shift
    return allocation

**workers + supervisor**

In [24]:
from langgraph_supervisor import create_supervisor
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

market_analysis_agent = create_agent(
    model=llm,
    tools=[get_stock_price, get_company_fundamentals, search_tool],
    name="market_analysis_agent",
)

portfolio_agent = create_agent(
    model=llm,
    tools=[propose_allocation],
    name="portfolio_agent",
)

compliance_agent = create_agent(
    model=llm,
    tools=[search_regulations],
    name="compliance_agent",
)

supervisor = create_supervisor(
    agents=[market_analysis_agent, portfolio_agent, compliance_agent],
    model=llm,
    prompt=(
        "You supervise a Market Analysis agent, a Portfolio Allocation agent, "
        "and a Compliance agent. Route stock/news questions to market_analysis_agent, "
        "allocation questions to portfolio_agent, and CMA-compliance questions to "
        "compliance_agent. Relay each worker's full answer to the user."
    ),
).compile(checkpointer=InMemorySaver())

**Testing**

In [25]:
config = {"configurable": {"thread_id": "router-test-1"}}
result = supervisor.invoke(
    {"messages": [{"role": "user", "content":
        "عندي 20000 ريال، مخاطرة متوسطة، وهدفي التقاعد. "
        "حلل السوق، اقترح توزيع، وتأكد إنه متوافق مع لوائح CMA."}]},
    config,
)

for m in result["messages"]:
    for tc in getattr(m, "tool_calls", []) or []:
        print("handoff ->", tc["name"])

print()
print(result["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 5.035982973s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTi

#  **Human-in-the-loop**

**Allocation Decision**

In [ ]:
from pydantic import BaseModel
from typing import Literal, Optional
from langgraph.types import interrupt, Command

class AllocationDecision(BaseModel):
    decision: Literal["approve", "reject", "modify"]
    modified_amount: Optional[float] = None
    note: Optional[str] = None

**human approval**

In [ ]:
from langgraph.types import interrupt

@tool
def human_approval(final_recommendation: str, compliance_notes: str, runtime: ToolRuntime) -> str:
    payload = {
        "action": "human_approval",
        "final_recommendation": final_recommendation,
        "compliance_notes": compliance_notes,
    }
    human_response = interrupt(payload)
    decision: AllocationDecision = human_response

    if decision.decision == "approve":
        return f"Approved: {final_recommendation}"
    elif decision.decision == "modify":
        return f"Modified to amount: {decision.modified_amount}"
    else:
        return f"Rejected. Note: {decision.note}"

**create agent**

In [ ]:
agent = create_agent(
    model=llm,
    tools=[record_risk_profile, human_approval],
    state_schema=InvestorState,
    checkpointer=checkpointer,
)

# **LangGraph functional API & error handling**

In [ ]:
import random
from typing import Dict, Any
from langgraph.func import task, entrypoint
from langgraph.types import interrupt, Command, RetryPolicy

In [ ]:
retriever = get_retriever()

fetch_retry_policy = RetryPolicy(
    retry_on=(ConnectionError, TimeoutError),
    max_attempts=3,
    initial_interval=1.0,
    backoff_factor=2.0
)

@task(retry=fetch_retry_policy)
def fetch_market_data_task(ticker: str) -> Dict[str, Any]:
    if random.random() < 0.1:
        raise ConnectionError("Connection failed")

    return get_stock_price.invoke({"ticker": ticker})

@task
def fetch_compliance_rag_task(query: str) -> Dict[str, Any]:
    try:
        docs = retriever.invoke(query)
        context_text = "\n\n".join([doc.page_content for doc in docs])
        status = "compliant" if context_text else "needs_review"
    except Exception:
        context_text = "Regulatory data unavailable. Flagged for manual audit."
        status = "system_fallback"

    return {
        "query": query,
        "retrieved_context": context_text,
        "status": status
    }

In [ ]:
import unittest.mock as mock

call_count = {"n": 0}

@task(retry=fetch_retry_policy)
def flaky_task_demo(ticker: str) -> dict:
    call_count["n"] += 1
    if call_count["n"] < 2:
        raise ConnectionError("Simulated transient failure")
    return get_stock_price.invoke({"ticker": ticker})

result = flaky_task_demo("AAPL").result()
print(f"Succeeded after {call_count['n']} attempt(s)")
print(result)

In [ ]:
@entrypoint(checkpointer=checkpointer)
def investment_advisor_workflow(inputs: dict, *, llm) -> dict:
    ticker = inputs["ticker"]
    risk_profile = inputs["risk_profile"]
    horizon = inputs["investment_horizon_years"]

    market_data = fetch_market_data_task(ticker).result()

    strategy = generate_strategy(risk_profile, horizon).result()
    trail = []
    verdict = None
    for round_num in range(1, MAX_OPTIMIZATION_ROUNDS + 1):
        verdict = evaluate_compliance(strategy, llm).result()
        trail.append({"round": round_num, "strategy": strategy, "verdict": verdict})
        if verdict.compliant:
            break
        strategy = optimize_strategy(strategy, verdict).result()

    approval_payload = {
        "action": "human_approval",
        "final_recommendation": strategy.asset_allocation,
        "compliance_notes": verdict.rationale,
        "market_summary": market_data,
    }
    human_response = interrupt(approval_payload)
    decision: AllocationDecision = human_response

    if decision.decision == "approve":
        approval_status = "Approved"
    elif decision.decision == "modify":
        approval_status = f"Modified to amount: {decision.modified_amount}"
    else:
        approval_status = "Rejected"

    return {
        "ticker": ticker,
        "market_data": market_data,
        "final_strategy": strategy,
        "final_verdict": verdict,
        "trail": trail,
        "approval_status": approval_status,
    }

In [ ]:
cfg = {"configurable": {"thread_id": "demo-investor-1"}}
result = investment_advisor_workflow.invoke(
    {"ticker": "AAPL", "risk_profile": "aggressive", "investment_horizon_years": 15},
    cfg, llm=llm,
)
print(result.get("__interrupt__"))

resumed = investment_advisor_workflow.invoke(
    Command(resume=AllocationDecision(decision="approve")),
    config=cfg, llm=llm,
)
print(f"Approval status: {resumed['approval_status']}")
print(f"Rounds used: {len(resumed['trail'])}")
print(f"Final allocation: {resumed['final_strategy'].asset_allocation}")

# **SQLite Persistence**

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string("checkpoints.db") as sqlite_checkpointer:
    persistent_agent = create_agent(
        model=llm,
        tools=[record_risk_profile, human_approval],
        state_schema=InvestorState,
        checkpointer=sqlite_checkpointer,
    )
    result = persistent_agent.invoke(
        {"messages": [HumanMessage("I prefer high risk and my goal is growth, record it")]},
        {"configurable": {"thread_id": "persistence-test"}}
    )
    for m in result["messages"]:
        m.pretty_print()